# 03 — Format & Structure Validators

33 examples covering ValidJSON, ValidPython, ValidSQL, ValidHTML, ValidURL,
ValidAddress, RegexMatch, ValidLength, ValidRange, ContainsString, EndsWith,
OneLine, and ValidChoices.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv
guardrails hub install hub://guardrails/valid_json
guardrails hub install hub://guardrails/valid_python
guardrails hub install hub://guardrails/valid_sql
guardrails hub install hub://guardrails/valid_html
guardrails hub install hub://guardrails/valid_url
guardrails hub install hub://guardrails/valid_address
guardrails hub install hub://guardrails/regex_match
guardrails hub install hub://guardrails/valid_length
guardrails hub install hub://guardrails/valid_range
guardrails hub install hub://guardrails/contains_string
guardrails hub install hub://guardrails/ends_with
guardrails hub install hub://guardrails/one_line
guardrails hub install hub://guardrails/valid_choices
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

In [ ]:
# Install Hub validators (run once)
!guardrails hub install hub://guardrails/valid_json --quiet
!guardrails hub install hub://guardrails/valid_python --quiet
!guardrails hub install hub://guardrails/valid_sql --quiet
!guardrails hub install hub://guardrails/valid_html --quiet
!guardrails hub install hub://guardrails/valid_url --quiet
!guardrails hub install hub://guardrails/valid_address --quiet
!guardrails hub install hub://guardrails/regex_match --quiet
!guardrails hub install hub://guardrails/valid_length --quiet
!guardrails hub install hub://guardrails/valid_range --quiet
!guardrails hub install hub://guardrails/contains_string --quiet
!guardrails hub install hub://guardrails/ends_with --quiet
!guardrails hub install hub://guardrails/one_line --quiet
!guardrails hub install hub://guardrails/valid_choices --quiet

## ValidJSON Examples (01–03)

In [ ]:
# Example 01: Malformed JSON blocked
from guardrails.hub import ValidJson
guard = Guard().use(ValidJson(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('{"name": "Alice", "age": 30')  # missing closing brace
except ValidationError:
    print('FAIL - malformed JSON blocked')

In [ ]:
# Example 02: Valid JSON string passes
from guardrails.hub import ValidJson
guard = Guard().use(ValidJson(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('{"name": "Alice", "age": 30, "active": true}')
print('PASS - valid JSON:', outcome.validation_passed)

In [ ]:
# Example 03: ValidJSON with FIX action (attempts repair)
from guardrails.hub import ValidJson
guard = Guard().use(ValidJson(on_fail=OnFailAction.FIX))
outcome = guard.validate('{"key": "value"')  # missing closing brace
print('FIX result:', outcome.validated_output)
print('passed:', outcome.validation_passed)

## ValidPython Examples (04–06)

In [ ]:
# Example 04: Python syntax error blocked
from guardrails.hub import ValidPython
guard = Guard().use(ValidPython(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('def foo(: pass')  # invalid syntax
except ValidationError:
    print('FAIL - Python syntax error blocked')

In [ ]:
# Example 05: Valid Python function passes
from guardrails.hub import ValidPython
guard = Guard().use(ValidPython(on_fail=OnFailAction.EXCEPTION))
valid_code = 'def add(a: int, b: int) -> int:\n    return a + b'
outcome = guard.validate(valid_code)
print('PASS - valid Python:', outcome.validation_passed)

In [ ]:
# Example 06: Indentation error detected
from guardrails.hub import ValidPython
guard = Guard().use(ValidPython(on_fail=OnFailAction.EXCEPTION))
bad_indent = 'def greet():\nprint("Hello")  # missing indent'
try:
    guard.validate(bad_indent)
except ValidationError:
    print('FAIL - indentation error detected')

## ValidSQL Examples (07–09)

In [ ]:
# Example 07: SQL with typo blocked
from guardrails.hub import ValidSql
guard = Guard().use(ValidSql(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('SELECT * FORM users WHERE active = 1')  # FORM instead of FROM
except ValidationError:
    print('FAIL - SQL syntax error blocked')

In [ ]:
# Example 08: Valid SELECT statement passes
from guardrails.hub import ValidSql
guard = Guard().use(ValidSql(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('SELECT id, name, email FROM users WHERE active = 1 ORDER BY name')
print('PASS - valid SELECT:', outcome.validation_passed)

In [ ]:
# Example 09: INSERT statement validation
from guardrails.hub import ValidSql
guard = Guard().use(ValidSql(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate("INSERT INTO products (name, price, stock) VALUES ('Widget', 9.99, 100)")
print('PASS - valid INSERT:', outcome.validation_passed)

## ValidHTML Examples (10–11)

In [ ]:
# Example 10: Unclosed HTML tag blocked
from guardrails.hub import ValidHtml
guard = Guard().use(ValidHtml(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('<div><p>Unclosed paragraph')  # missing </p></div>
except ValidationError:
    print('FAIL - unclosed HTML tag blocked')

In [ ]:
# Example 11: Valid minimal HTML passes
from guardrails.hub import ValidHtml
guard = Guard().use(ValidHtml(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('<html><body><p>Hello, World!</p></body></html>')
print('PASS - valid HTML:', outcome.validation_passed)

## ValidURL Examples (12–14)

In [ ]:
# Example 12: Malformed URL blocked
from guardrails.hub import ValidUrl
guard = Guard().use(ValidUrl(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('htp://example')  # malformed scheme
except ValidationError:
    print('FAIL - malformed URL blocked')

In [ ]:
# Example 13: Valid HTTPS URL passes
from guardrails.hub import ValidUrl
guard = Guard().use(ValidUrl(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('https://www.guardrailsai.com/docs')
print('PASS - valid HTTPS URL:', outcome.validation_passed)

In [ ]:
# Example 14: FTP and other URL schemes
from guardrails.hub import ValidUrl
guard = Guard().use(ValidUrl(on_fail=OnFailAction.NOOP))
for url in ['ftp://files.example.com/data.csv', 'mailto:user@example.com', 'not-a-url']:
    outcome = guard.validate(url)
    print(f'  {url:<45}  passed={outcome.validation_passed}')

## ValidAddress Examples (15–16)

In [ ]:
# Example 15: Non-existent/fake address blocked
from guardrails.hub import ValidAddress
guard = Guard().use(ValidAddress(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('123 Xyz Nonexistent Blvd, FakeTown, ZZ 00000')
except ValidationError:
    print('FAIL - invalid address blocked')

In [ ]:
# Example 16: Well-known real US address passes
from guardrails.hub import ValidAddress
guard = Guard().use(ValidAddress(on_fail=OnFailAction.NOOP))
outcome = guard.validate('1600 Pennsylvania Ave NW, Washington, DC 20500')
print('White House address result:', outcome.validation_passed)

## RegexMatch Examples (17–19)

In [ ]:
# Example 17: Phone number format validation (pass + fail)
from guardrails.hub import RegexMatch
guard = Guard().use(RegexMatch(regex=r'^\+1\d{10}$', on_fail=OnFailAction.EXCEPTION))
for phone in ['+12125551234', '212-555-1234', '+1212555ABCD']:
    try:
        guard.validate(phone)
        print(f'  PASS: {phone}')
    except ValidationError:
        print(f'  FAIL: {phone}')

In [ ]:
# Example 18: Email format regex validation
from guardrails.hub import RegexMatch
email_regex = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
guard = Guard().use(RegexMatch(regex=email_regex, on_fail=OnFailAction.EXCEPTION))
for email in ['user@example.com', 'invalid-email', 'also@bad']:
    try:
        guard.validate(email)
        print(f'  PASS: {email}')
    except ValidationError:
        print(f'  FAIL: {email}')

In [ ]:
# Example 19: UUID format validation
from guardrails.hub import RegexMatch
uuid_regex = r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$'
guard = Guard().use(RegexMatch(regex=uuid_regex, on_fail=OnFailAction.EXCEPTION))
valid_uuid = '550e8400-e29b-41d4-a716-446655440000'
outcome = guard.validate(valid_uuid)
print('PASS - UUID valid:', outcome.validation_passed)

## ValidLength Examples (20–22)

In [ ]:
# Example 20: Response too short (min length enforcement)
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=100, max=2000, on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Too brief.')  # only 10 chars
except ValidationError:
    print('FAIL - response too short')

In [ ]:
# Example 21: Response too long (max length enforcement)
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=0, max=50, on_fail=OnFailAction.EXCEPTION))
long_response = 'This is a very long response that exceeds the maximum allowed length of fifty characters.'
try:
    guard.validate(long_response)
except ValidationError:
    print('FAIL - response too long:', len(long_response), 'chars')

In [ ]:
# Example 22: Response in valid range passes
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=20, max=300, on_fail=OnFailAction.EXCEPTION))
response = 'The sky is blue due to Rayleigh scattering of sunlight.'
outcome = guard.validate(response)
print(f'PASS - {len(response)} chars in [20,300]:', outcome.validation_passed)

## ValidRange Examples (23–24)

In [ ]:
# Example 23: Numeric value out of range blocked
from guardrails.hub import ValidRange
guard = Guard().use(ValidRange(min=0, max=1000, on_fail=OnFailAction.EXCEPTION))
for price in ['500', '-10', '1500']:
    try:
        guard.validate(price)
        print(f'  PASS: price={price}')
    except ValidationError:
        print(f'  FAIL: price={price} out of [0,1000]')

In [ ]:
# Example 24: Float confidence score must be 0.0–1.0
from guardrails.hub import ValidRange
guard = Guard().use(ValidRange(min=0.0, max=1.0, on_fail=OnFailAction.EXCEPTION))
for score in ['0.75', '0.0', '1.0', '1.5', '-0.1']:
    try:
        guard.validate(score)
        print(f'  PASS: score={score}')
    except ValidationError:
        print(f'  FAIL: score={score} out of [0.0,1.0]')

## ContainsString Examples (25–26)

In [ ]:
# Example 25: Required legal disclaimer keyword absent — blocked
from guardrails.hub import ContainsString
guard = Guard().use(ContainsString(search_string='disclaimer', on_fail=OnFailAction.EXCEPTION))
legal_text = 'Our investment products carry significant risk and may lose value.'
try:
    guard.validate(legal_text)  # missing 'disclaimer'
except ValidationError:
    print('FAIL - required keyword "disclaimer" absent')

In [ ]:
# Example 26: Required keyword present — passes
from guardrails.hub import ContainsString
guard = Guard().use(ContainsString(search_string='consult a professional', on_fail=OnFailAction.EXCEPTION))
response = 'Before making any investment, please consult a professional financial advisor.'
outcome = guard.validate(response)
print('PASS - required phrase present:', outcome.validation_passed)

## EndsWith Examples (27–28)

In [ ]:
# Example 27: Sentence does not end with period — blocked
from guardrails.hub import EndsWith
guard = Guard().use(EndsWith(end='.', on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('The answer is forty-two')  # no period
except ValidationError:
    print('FAIL - does not end with period')

In [ ]:
# Example 28: Response ends with correct character — passes
from guardrails.hub import EndsWith
guard = Guard().use(EndsWith(end='?', on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('Are you sure you want to proceed?')
print('PASS - ends with question mark:', outcome.validation_passed)

## OneLine Examples (29–30)

In [ ]:
# Example 29: Multi-line response blocked when OneLine is required
from guardrails.hub import OneLine
guard = Guard().use(OneLine(on_fail=OnFailAction.EXCEPTION))
multi = 'First line.\nSecond line.\nThird line.'
try:
    guard.validate(multi)
except ValidationError:
    print('FAIL - multi-line response blocked')

In [ ]:
# Example 30: Single clean line passes OneLine
from guardrails.hub import OneLine
guard = Guard().use(OneLine(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('The capital of France is Paris.')
print('PASS - single line:', outcome.validation_passed)

## ValidChoices Examples (31–33)

In [ ]:
# Example 31: Model outputs value not in allowed set — blocked
from guardrails.hub import ValidChoices
guard = Guard().use(ValidChoices(choices=['red', 'blue', 'green'], on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('purple')  # not in allowed choices
except ValidationError:
    print('FAIL - "purple" not in [red, blue, green]')

In [ ]:
# Example 32: Case sensitivity — exact match required
from guardrails.hub import ValidChoices
guard = Guard().use(ValidChoices(choices=['positive', 'negative', 'neutral'], on_fail=OnFailAction.NOOP))
for val in ['positive', 'Positive', 'POSITIVE', 'neutral']:
    outcome = guard.validate(val)
    print(f'  {val!r:<12} passed={outcome.validation_passed}')

In [ ]:
# Example 33: ValidChoices with REASK — LLM re-prompted to pick from valid list
from guardrails.hub import ValidChoices
choices = ['low', 'medium', 'high', 'critical']
guard = Guard().use(ValidChoices(choices=choices, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt=f'Rate the priority of a production outage. Choose ONLY one of: {choices}. Reply with the single word only.',
    model=MODEL,
    num_reasks=2
)
print('REASK result:', outcome.validated_output)
print('valid choice?', outcome.validated_output in choices if outcome.validated_output else 'N/A')